# Lab 41: The Invisible Push

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 5, *The Memory Hallucination*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-41.ipynb)

**What you will do:** test the food/cleaning priming puzzle on your friends, check whether their completions beat chance, pool the class's results, and see whether a small language model shifts its own most-likely word choice the same way when it is "primed" with a food or cleaning context.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 41 you split several friends into two groups, had each group read a short paragraph
about either food or cleaning, then had everyone complete an ambiguous fragment such as
"S O _ P" — which can become SOUP or SOAP. People primed with food tend to write SOUP; people
primed with cleaning tend to write SOAP, and most say afterwards that they felt their answer
was spontaneous, not influenced by the paragraph at all.

## Record your results

Test the puzzle on as many friends as you can, alternating which paragraph each one reads.
Record each trial as a row: which priming condition they got, and which word they wrote for
the fragment "S O _ P". The rows below are example trials so the cell runs — replace them
with your own.

In [ ]:
# example data: replace with your own trials
trial_data = pd.DataFrame({
    "participant": ["Friend A", "Friend B", "Friend C", "Friend D"],
    "condition": ["food", "food", "cleaning", "cleaning"],
    "completed_word": ["soup", "soup", "soap", "soup"],
})  # example data
trial_data["primed_consistent"] = ((trial_data.condition == "food") & (trial_data.completed_word == "soup")) | \
                                   ((trial_data.condition == "cleaning") & (trial_data.completed_word == "soap"))
display(trial_data)

rate_by_condition = trial_data.groupby("condition")["primed_consistent"].mean()
display(rate_by_condition.to_frame("proportion primed-consistent"))
rate_by_condition.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3))
plt.ylim(0, 1); plt.ylabel("proportion primed-consistent"); plt.show()

## Compare

The book does not give a published percentage for how often a SOUP/SOAP-style completion
follows the matching prime — only the qualitative finding that people primed with food are
"more likely" to write SOUP, and people primed with cleaning are "more likely" to write
SOAP. The natural benchmark, if the paragraph had no effect at all, is chance: 50%, since
both words are equally valid completions of the fragment.

In [ ]:
overall_rate = trial_data.primed_consistent.mean()
print(f"Your overall proportion primed-consistent: {overall_rate:.0%} (chance = 50%)")
if overall_rate > 0.5:
    print("Above chance -- consistent with the direction the spreading-activation account (Meyer & Schvaneveldt, 1971) predicts.")
elif overall_rate == 0.5:
    print("Exactly at chance -- with only a few trials that is not surprising; priming effects are usually")
    print("measured across many participants, not a handful.")
else:
    print("Below chance in this small sample. With only a few trials, a couple of contrarian friends can")
    print("easily push the rate under 50%; this is exactly why psychologists run the paradigm on many participants.")

## The AI side

The book's AI Connection compares priming to the way a prompt shapes a language model's
output, and to **few-shot prompting** — giving a model a few examples that establish a
pattern it then continues (Brown et al., 2020, *Advances in Neural Information Processing
Systems*). Below, a small open language model is given the same word-completion puzzle
embedded in two short stories: one about cooking, one about cleaning. Rather than just
generating free text (which a small model can do unreliably), we measure which of the two
candidate words — SOUP or SOAP — the model finds more probable as a continuation, under
each story.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

def logprob_of_continuation(prompt, continuation):
    ids_prompt = tokenizer(prompt, return_tensors="pt").input_ids
    ids_full = tokenizer(prompt + continuation, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids_full).logits
    logprobs = torch.log_softmax(logits, dim=-1)
    total = 0.0
    for i in range(ids_prompt.shape[1], ids_full.shape[1]):
        total += logprobs[0, i - 1, ids_full[0, i]].item()
    return total

food_context = ("Maria spent the whole morning in the kitchen, chopping vegetables and stirring "
                 "a big pot of soup on the stove.")
clean_context = ("Maria spent the whole morning in the bathroom, scrubbing the tiles and washing "
                  "the floor with soap and a brush.")
puzzle = " Later, she was given the letters S, O, a blank, and P and asked to complete the word. She wrote the word"
candidates = [" SOUP", " SOAP"]

rows = []
for label, context in [("food-primed", food_context), ("cleaning-primed", clean_context)]:
    lps = {c.strip(): logprob_of_continuation(context + puzzle, c) for c in candidates}
    winner = max(lps, key=lps.get)
    rows.append({"priming context": label, "log-prob(SOUP)": round(lps["SOUP"], 2),
                 "log-prob(SOAP)": round(lps["SOAP"], 2), "model's more likely word": winner})
ai = pd.DataFrame(rows)
display(ai)

If the model's more likely word tracks the priming context — SOUP after the cooking story,
SOAP after the cleaning story — the model is showing the same context-driven shift in what
comes to mind that your friends showed. The mechanism is different: your friends' brains
spread activation through an associative network of meanings; the model shifted the
conditional probability of one word over another given the preceding tokens, with no
associative network and no meaning in the human sense at all. Because `distilgpt2` is a
small, general-purpose model rather than one built for this task, the shift may be modest,
or it may occasionally point the same way regardless of context — unlike the psychological
effect, which is well replicated across many participants, a single tiny model on a single
item is a weak test, just as a single friend's answer is a weak test of the human effect.

## Pool the class data

Pool completions across the class: each pair adds one row (an anonymous participant ID,
the priming condition they were given, and which word they wrote). The example rows below
are placeholders so the cell runs.

In [ ]:
import io
csv_text = """participant_id,condition,completed_word
P01,food,soup
P02,food,soup
P03,food,soap
P04,cleaning,soap
P05,cleaning,soap
P06,cleaning,soup
P07,food,soup
P08,cleaning,soap
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
class_df["primed_consistent"] = ((class_df.condition == "food") & (class_df.completed_word == "soup")) | \
                                 ((class_df.condition == "cleaning") & (class_df.completed_word == "soap"))
rate_by_condition = class_df.groupby("condition")["primed_consistent"].mean()
display(rate_by_condition.to_frame("proportion primed-consistent"))

boot = [class_df.sample(len(class_df), replace=True)["primed_consistent"].mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Overall proportion primed-consistent: {class_df.primed_consistent.mean():.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(class_df)})")
print("Chance, if the paragraph had no effect at all, would be 50%.")

rate_by_condition.plot.bar(color=["steelblue", "firebrick"], figsize=(5, 3))
plt.axhline(0.5, color="grey", linestyle="--", label="chance")
plt.ylabel("proportion primed-consistent"); plt.legend(); plt.ylim(0, 1); plt.show()

## Questions to think about

1. Did your friends' completions land above the 50% chance level expected if the paragraph had no effect? With so few trials, how much would you trust a single result either way?
2. In the ai_lab demo, did the model's more likely word track the priming context? What differs about *how* the human brain and the language model each arrived at a context-dependent answer, even where the outcome looks similar?
3. Repetition suppression is described in the text as neurons needing "less activation to fire" after a related prime, experienced subjectively as fluency. Which part of your record or ai_lab data would you look at if you wanted evidence of fluency, rather than just the final word chosen?
4. The text distinguishes priming (adjusting existing associations) from few-shot prompting (establishing a brand-new pattern from a couple of examples). Is the food/cleaning paragraph in this Lab closer to priming or to few-shot prompting, or does it not map cleanly onto either? Explain your reasoning.

## Challenge

Design a new pair of priming paragraphs around a different ambiguous fragment of your own
choosing (one that could plausibly complete two different ways depending on context) and test
it on at least three friends.

In [ ]:
# Example data: replace with your own fragment, conditions and completions.
my_fragment_description = "choose your own ambiguous fragment, e.g. one that could complete as a sport word or a music word"  # example
my_trials = pd.DataFrame({
    "participant": ["Friend E", "Friend F", "Friend G"],
    "condition": ["A", "A", "B"],           # example data: which priming paragraph each friend read
    "primed_consistent": [True, True, False],  # example data: did their completion match their own condition's prime?
})  # example data

print(f"Fragment used: {my_fragment_description}")
print(my_trials)
rate = my_trials["primed_consistent"].mean()
print(f"Overall proportion primed-consistent: {rate:.0%} (chance, with no priming effect, is still 50%)")
print("The values above are placeholders -- replace the fragment, conditions and completions with your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-41.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")